# Архитектура моделей в PyTorch: <br>nn.Module и контейнеры

В предыдущем видео мы изучили отдельные слои. Но нейросеть — это композиция <br>слоев.
В PyTorch базовым строительным блоком для любой модели является <br>класс `nn.Module`.

В этом ноутбуке мы разберем:
1. Как правильно создать свой класс модели (`nn.Module`).
2. Когда использовать `nn.Sequential` (и его ограничения).
3. **Критически важную разницу** между обычным списком Python `[]` и <br>`nn.ModuleList`.
4. Как использовать `nn.ModuleDict` для сложных архитектур с ветвлением.

In [3]:
import torch
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Доступное устройство: {device}")

Доступное устройство: cpu


## 1. Базовый класс nn.Module
Любая нейронная сеть в PyTorch должна наследоваться от `nn.Module`.

Класс для создания нейронной сети обязан иметь два метода:
- `__init__`: здесь мы **объявляем** и инициализируем слои (как свойства класса `self`).
- `forward`: здесь мы описываем **логику прохождения данных** (прямой проход) <br>через эти слои.

In [2]:
class SimpleNN(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super().__init__() # Обязательно вызывать конструктор родителя!

        # 1. Объявляем слои
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, num_classes)
        self.relu = nn.ReLU()

    def forward(self, x):
        # 2. Описываем прямой проход
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Создаём модель
model = SimpleNN(input_size=3, hidden_size=5, num_classes=2)
print("Структура модели:")
print(model)

# Тестируем модель
dummy_input = torch.randn(4, 3) # Batch size = 4

output = model(dummy_input) # Вызываем как функцию!
print(f"\nРазмер выхода: {output.shape}")   # Ожидаем: [4, 2]


Структура модели:
SimpleNN(
  (fc1): Linear(in_features=3, out_features=5, bias=True)
  (fc2): Linear(in_features=5, out_features=2, bias=True)
  (relu): ReLU()
)

Размер выхода: torch.Size([4, 2])


In [3]:
# Доступ к конкретному слою по имени:
print("Доступ к слою 'fc2':")
print(model.fc2)

Доступ к слою 'fc2':
Linear(in_features=5, out_features=2, bias=True)


In [4]:
# Все обучаемые параметры модели
for param in model.parameters():
    print(param)

Parameter containing:
tensor([[-0.1246, -0.0209,  0.1728],
        [-0.5296,  0.1543, -0.3585],
        [-0.4041,  0.0225,  0.0968],
        [-0.2266, -0.1669,  0.4905],
        [ 0.5240,  0.5144,  0.1238]], requires_grad=True)
Parameter containing:
tensor([ 0.2115,  0.4256, -0.0354,  0.4472, -0.2060], requires_grad=True)
Parameter containing:
tensor([[ 0.0420, -0.0239,  0.2171,  0.2711, -0.2894],
        [ 0.2493, -0.1922,  0.4413,  0.2557, -0.0522]], requires_grad=True)
Parameter containing:
tensor([-0.2383,  0.2219], requires_grad=True)


In [5]:
# Все слои модели
for chl in model.children():
    print(chl)

Linear(in_features=3, out_features=5, bias=True)
Linear(in_features=5, out_features=2, bias=True)
ReLU()


In [6]:
for name_param in model.named_parameters():
    print(name_param)

('fc1.weight', Parameter containing:
tensor([[-0.1246, -0.0209,  0.1728],
        [-0.5296,  0.1543, -0.3585],
        [-0.4041,  0.0225,  0.0968],
        [-0.2266, -0.1669,  0.4905],
        [ 0.5240,  0.5144,  0.1238]], requires_grad=True))
('fc1.bias', Parameter containing:
tensor([ 0.2115,  0.4256, -0.0354,  0.4472, -0.2060], requires_grad=True))
('fc2.weight', Parameter containing:
tensor([[ 0.0420, -0.0239,  0.2171,  0.2711, -0.2894],
        [ 0.2493, -0.1922,  0.4413,  0.2557, -0.0522]], requires_grad=True))
('fc2.bias', Parameter containing:
tensor([-0.2383,  0.2219], requires_grad=True))


In [7]:
# Словарь, содержащий ссылки на все состояние модели
# Включает в себя обучаемые параметры и буферы
model.state_dict()

OrderedDict([('fc1.weight',
              tensor([[-0.1246, -0.0209,  0.1728],
                      [-0.5296,  0.1543, -0.3585],
                      [-0.4041,  0.0225,  0.0968],
                      [-0.2266, -0.1669,  0.4905],
                      [ 0.5240,  0.5144,  0.1238]])),
             ('fc1.bias',
              tensor([ 0.2115,  0.4256, -0.0354,  0.4472, -0.2060])),
             ('fc2.weight',
              tensor([[ 0.0420, -0.0239,  0.2171,  0.2711, -0.2894],
                      [ 0.2493, -0.1922,  0.4413,  0.2557, -0.0522]])),
             ('fc2.bias', tensor([-0.2383,  0.2219]))])

In [8]:
# Все буферы модели
for bf in model.buffers():
    print(bf)

**ОДНА ИЗ САМЫХ ЧАСТЫХ ОШИБОК** - забывают перевести модель в нужный режим.

In [9]:
# Переводим модель в режим обучения
model.train()

# Переводим модель в режим валидации\предсказания
model.eval()

SimpleNN(
  (fc1): Linear(in_features=3, out_features=5, bias=True)
  (fc2): Linear(in_features=5, out_features=2, bias=True)
  (relu): ReLU()
)

In [10]:
# Переводим модель на разные устройства
model.to(device)

SimpleNN(
  (fc1): Linear(in_features=3, out_features=5, bias=True)
  (fc2): Linear(in_features=5, out_features=2, bias=True)
  (relu): ReLU()
)

## 2. nn.Sequential
Это контейнер, который применяет слои строго по порядку.

Идеально подходит для простых стеков (например, блок **Conv -> BN -> ReLU**).

**Ограничения:** В `Sequential` нельзя сделать skip-connections (как в ResNet) или <br>использовать одни и те же данные в разных ветках.

In [11]:
# Способ 1: Передача слоев как аргументов
seq_model_1 = nn.Sequential(
    nn.Linear(10, 32),
    nn.ReLU(),
    nn.Linear(32, 2)
)

print("Структура модели:")
print(seq_model_1)

Структура модели:
Sequential(
  (0): Linear(in_features=10, out_features=32, bias=True)
  (1): ReLU()
  (2): Linear(in_features=32, out_features=2, bias=True)
)


In [12]:
# Доступ к конкретному слою по индексу:
print("Доступ к слою с индексам idx=0':")
print(seq_model_1[0])

Доступ к слою с индексам idx=0':
Linear(in_features=10, out_features=32, bias=True)


In [13]:
# Способ 2: Передача через OrderedDict
# Позволяет дать слоям имена, что критически важно для отладки и сохранения весов
from collections import OrderedDict

seq_model_2 = nn.Sequential(OrderedDict([
    ('feature_extractor', nn.Linear(10, 32)),
    ('activation', nn.ReLU()),
    ('classifier', nn.Linear(32, 2))
]))

print("Модель с именами слоев:")
print(seq_model_2)


Модель с именами слоев:
Sequential(
  (feature_extractor): Linear(in_features=10, out_features=32, bias=True)
  (activation): ReLU()
  (classifier): Linear(in_features=32, out_features=2, bias=True)
)


In [14]:
# Доступ к конкретному слою по имени:
print("\nДоступ к слою 'classifier':")
print(seq_model_2.classifier)


Доступ к слою 'classifier':
Linear(in_features=32, out_features=2, bias=True)


## 3. nn.ModuleList (Частая ошибка новичков!)
Что делать, если количество слоев заранее неизвестно или их много (например, <br>10 одинаковых блоков)?

Кажется логичным использовать обычный список Python `[]`. **НЕ ДЕЛАЙТЕ ЭТОГО!**

Обычный список Python не регистрирует параметры внутри `nn.Module`. <br>Оптимизатор их "не увидит".

Для этого существует `nn.ModuleList`.

In [15]:
# ❌ ПЛОХОЙ ПРИМЕР (Обычный список Python)
class BadModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = [nn.Linear(10, 10), nn.Linear(10, 10)] # Обычный список!

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

bad_model = BadModel()
print("Параметры в 'плохой' модели:", list(bad_model.parameters()))
# ВЫВОД БУДЕТ ПУСТЫМ! [] Оптимизатор не сможет обучать эту модель.


# ✅ ХОРОШИЙ ПРИМЕР (nn.ModuleList)
class GoodModel(nn.Module):
    def __init__(self, num_layers):
        super().__init__()
        # Оборачиваем список слоев в nn.ModuleList
        self.layers = nn.ModuleList(
            [nn.Linear(10, 10) for _ in range(num_layers)]
        )

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

good_model = GoodModel(num_layers=10)
print(
    f"\nКоличество параметров в 'хорошей' модели: "
    f"{sum(p.numel() for p in good_model.parameters())}"
)
# ВЫВОД: Параметры успешно зарегистрированы и посчитаны!

Параметры в 'плохой' модели: []

Количество параметров в 'хорошей' модели: 1100


In [16]:
# Посмотрим какие слои содердит модель
good_model

GoodModel(
  (layers): ModuleList(
    (0-9): 10 x Linear(in_features=10, out_features=10, bias=True)
  )
)

## 4. nn.ModuleDict
Аналогично `ModuleList`, но работает как словарь.

Полезно, когда у вас есть разные ветки обработки или набор инструментов, <br>к которым нужно обращаться по строковому ключу.

In [17]:
class MultiHeadModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Общий feature extractor
        self.feature_extractor = nn.Linear(10, 16)

        # Разные "головы" для разных задач
        self.heads = nn.ModuleDict({
            'classification': nn.Linear(16, 10),
            'regression': nn.Linear(16, 1),
            'detection': nn.Linear(16, 4)
        })

    def forward(self, x, task='classification'):
        x = self.feature_extractor(x)
        # Динамический выбор слоя по имени
        return self.heads[task](x)

multi_model = MultiHeadModel()
x = torch.randn(4, 10)

out_class = multi_model(x, task='classification')
out_reg   = multi_model(x, task='regression')
out_detc  = multi_model(x, task='detection')

print(f"Выход классификации: {out_class.shape}")    # [4, 2]
print(f"Выход регрессии:     {out_reg.shape}")      # [4, 1]
print(f"Выход детекции:      {out_detc.shape}")     # [4, 4]


Выход классификации: torch.Size([4, 10])
Выход регрессии:     torch.Size([4, 1])
Выход детекции:      torch.Size([4, 4])


In [18]:
print("\nСтруктура ModuleDict:")
print(multi_model.heads)



Структура ModuleDict:
ModuleDict(
  (classification): Linear(in_features=16, out_features=10, bias=True)
  (regression): Linear(in_features=16, out_features=1, bias=True)
  (detection): Linear(in_features=16, out_features=4, bias=True)
)


## 5. Практический пример: Комбинация подходов
В реальных проектах мы часто комбинируем всё вышесказанное:
- Наследуемся от `nn.Module`.
- Используем `nn.Sequential` внутри для линейных блоков.
- Используем `nn.ModuleList` для повторяющихся блоков.

In [19]:
class AdvancedCNN(nn.Module):
    def __init__(self, num_blocks=3):
        super().__init__()

        # 1. Начальный слой
        self.conv1 = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.BatchNorm2d(16),
            nn.ReLU()
        )

        # 2. Динамическое количество блоков (используем ModuleList)
        self.blocks = nn.ModuleList()
        for _ in range(num_blocks):
            block = nn.Sequential(
                nn.Conv2d(16, 16, kernel_size=3, stride=2, padding=1),
                nn.BatchNorm2d(16),
                nn.ReLU(),
            )
            self.blocks.append(block)

        # 3. Разные "головы" для разных задач
        # После 3-х слоёв размер с 64x64 уменьшится до 8x8
        self.heads = nn.ModuleDict({
            'classification': nn.Linear(16 * 8 * 8, 10),
            'regression': nn.Linear(16 * 8 * 8, 1),
            'detection': nn.Linear(16 * 8 * 8, 4)
        })

    def forward(self, x, task='classification'):
        x = self.conv1(x)

        # Проходим через все блоки
        for block in self.blocks:
            x = block(x)

        # Flatten (выпрямляем тензор для Linear слоя)
        # x.shape был [B, 16, 8, 8], станет [B, 16*8*8]
        x = torch.flatten(x, start_dim=1)

        return self.heads[task](x)

# Тест
final_model = AdvancedCNN(num_blocks=3)
img_batch = torch.randn(2, 3, 64, 64)    # 2 изображения 64x64, с 3 цветовыми каналами
out = final_model(img_batch, task='detection')

print(f"Итоговый выход: {out.shape}")    # [2, 4]


Итоговый выход: torch.Size([2, 4])


In [20]:
print("\nПолная архитектура:")
print(final_model)


Полная архитектура:
AdvancedCNN(
  (conv1): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
  )
  (blocks): ModuleList(
    (0-2): 3 x Sequential(
      (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (1): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): ReLU()
    )
  )
  (heads): ModuleDict(
    (classification): Linear(in_features=1024, out_features=10, bias=True)
    (regression): Linear(in_features=1024, out_features=1, bias=True)
    (detection): Linear(in_features=1024, out_features=4, bias=True)
  )
)


Вместо класса Sequential создаём свой универсальный класс Conv

In [ ]:
class Conv(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size=3,
        stride=1,
        padding=0,
        bn=True,
        act=None
    ):
        super().__init__()

        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            bias=not bn
        )
        self.bn = nn.BatchNorm2d(out_channels) if bn else nn.Identity()
        self.act = act() if act is not None else nn.ReLU(inplace=True)

    def forward(self, x):
            
            x = self.conv(x)
            x = self.bn(x)
            out = self.act(x)
    
            return out


class AdvancedCNN(nn.Module):
    def __init__(self, num_blocks=3):
        super().__init__()

        # 1. Начальный слой
        self.conv1 = Conv(3, 16, padding=1)

        # 2. Динамическое количество блоков (используем ModuleList)
        self.blocks = nn.ModuleList()
        for _ in range(num_blocks):
            block = Conv(16, 16, stride=2, padding=1)
            self.blocks.append(block)

        # 3. Разные "головы" для разных задач
        # После 3-х слоёв размер с 64x64 уменьшится до 8x8
        self.heads = nn.ModuleDict({
            'classification': nn.Linear(16 * 8 * 8, 10),
            'regression': nn.Linear(16 * 8 * 8, 1),
            'detection': nn.Linear(16 * 8 * 8, 4)
        })

    def forward(self, x, task='classification'):
        
        x = self.conv1(x)

        # Проходим через все блоки
        for block in self.blocks:
            x = block(x)

        # Flatten (выпрямляем тензор для Linear слоя)
        # x.shape был [B, 16, 8, 8], станет [B, 16*8*8]
        x = torch.flatten(x, start_dim=1)

        return self.heads[task](x)

# Тест
final_model = AdvancedCNN(num_blocks=3)
img_batch = torch.randn(2, 3, 64, 64)    # 2 изображения 64x64, с 3 цветовыми каналами
out = final_model(img_batch, task='detection')

print(f"Итоговый выход: {out.shape}")    # [2, 4]


Итоговый выход: torch.Size([2, 4])


In [12]:
print("\nПолная архитектура:")
print(final_model)


Полная архитектура:
AdvancedCNN(
  (conv1): Conv(
    (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (act): ReLU(inplace=True)
  )
  (blocks): ModuleList(
    (0-2): 3 x Conv(
      (conv): Conv2d(16, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (bn): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (act): ReLU(inplace=True)
    )
  )
  (heads): ModuleDict(
    (classification): Linear(in_features=1024, out_features=10, bias=True)
    (regression): Linear(in_features=1024, out_features=1, bias=True)
    (detection): Linear(in_features=1024, out_features=4, bias=True)
  )
)


## Итоги видео:
1. **Всегда** наследуйте свои модели от `nn.Module` и вызывайте `super().__init__()`.
3. **Никогда** не храните слои в обычном списке Python `[]` внутри `nn.Module`.<br>Используйте `nn.ModuleList`.
4. Используйте `nn.ModuleDict`, когда нужно обращаться к слоям по строковым ключам.
5. Метод `forward` определяет, как данные проходят через объявленные слои.

**Домашнее задание:** Попробуйте создать модель, которая принимает на вход <br>два разных тензора (например, изображение и числовой вектор), обрабатывает их <br>в разных ветках (`ModuleDict`), а затем конкатенирует (`torch.cat`) результаты <br>перед финальным `Linear` слоем.